# **FOREWORD**

This is a base inference starter using the artefacts from [here](https://www.kaggle.com/code/antoinemasq/birdclef-2026-pytorch-baseline-inference?scriptVersionId=302964055) and [here](https://www.kaggle.com/code/antoinemasq/birdclef-2026-pytorch-baseline-training) for the Birdclef 2026 competition. <br>
Thanks to the original work for the starter kernel <br>

This competition uses a modified AUC metric [here](https://www.kaggle.com/code/metric/birdclef-roc-auc) that needs to be maximized. 

**Why do we use OpenVINO:-** <br>

1. Lower latency: Hardware-specific graph optimizations (layer fusion, quantization) reduce CPU inference time significantly vs vanilla PyTorch.
2. Zero framework overhead: Compiled IR models skip Python/autograd overhead, running lean native kernels.
3. Cross-hardware portability: Same .xml/.bin runs on CPU, iGPU, VPU or NPU with a single device flag change.
4. Memory efficiency: Static graph + INT8 quantization support cuts memory footprint vs FP32 PyTorch models.

This kernel may be used as a starter to use this inference option with scripts to ensure smooth and easy blending/ experiment tracking going ahead. One may choose to augment and modify this as per one's own peril!


# **PUBLIC BASELINE**

In [ ]:
%%writefile myinfer.py

from warnings import filterwarnings
filterwarnings("ignore")

import os
import math
import timm
import torch
import torch.nn as nn
import torchaudio
import torchvision
import numpy as np
import pandas as pd
import openvino as ov
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor
from torch.utils.data import Dataset, DataLoader

# ─── Configuration ────────────────────────────────────────────────────────────

batch_size    = 64
ov_device     = "CPU"

PATH          = '/kaggle/input/competitions/birdclef-2026/'
TEST_PATH     = PATH + 'test_soundscapes/'
TRAIN_PATH    = PATH + 'train_soundscapes/'
taxonomy_csv  = PATH + 'taxonomy.csv'
SUBMISSION_FILE = "submission.csv"
model_dir       = "/kaggle/input/notebooks/antoinemasq/birdclef-2026-pytorch-baseline-training/models"

DUR  = 5
SR   = 32000

# ─── Spectrogram ─────────────────────────────────────────────────────────────

class Spectrogram(nn.Module):
    def __init__(self, sr=32000, n_fft=2048, n_mels=256, hop_length=512,
                 f_min=20, f_max=16000, channels=1, norm="slaney",
                 mel_scale="htk", target_size=(256, 256), top_db=80.0, **kwargs):
        super().__init__()
        self.channels = channels
        self.top_db   = top_db
        self.mel_transform = torchaudio.transforms.MelSpectrogram(
            sample_rate=sr, n_fft=n_fft, hop_length=hop_length,
            n_mels=n_mels, f_min=f_min, f_max=f_max,
            mel_scale=mel_scale, pad_mode="reflect", power=2.0,
            norm=norm, center=True,
        )
        self.resize = torchvision.transforms.Resize(size=target_size)

    def power_to_db(self, S):
        amin    = 1e-10
        log_spec = 10.0 * torch.log10(S.clamp(min=amin))
        log_spec -= 10.0 * torch.log10(torch.tensor(amin).to(S))
        if self.top_db is not None:
            max_val  = log_spec.flatten(-2).max(dim=-1).values[..., None, None]
            log_spec = torch.maximum(log_spec, max_val - self.top_db)
        return log_spec

    def forward(self, x, resize=True):
        squeeze = x.dim() == 1
        if squeeze:
            x = x.unsqueeze(0)
        mel = self.mel_transform(x)
        mel = self.power_to_db(mel)
        mel = mel.unsqueeze(1).repeat(1, self.channels, 1, 1)
        if resize:
            mel = self.resize(mel)
        B, C = mel.shape[:2]
        flat = mel.view(B, C, -1)
        mins = flat.min(dim=-1).values[..., None, None]
        maxs = flat.max(dim=-1).values[..., None, None]
        mel  = (mel - mins) / (maxs - mins + 1e-7)
        if squeeze:
            mel = mel.squeeze(0)
        return mel

# ─── Model ───────────────────────────────────────────────────────────────────

class BirdModel(nn.Module):
    def __init__(self, config=None):
        super().__init__()
        cfg = {
            'backbone':        'tf_efficientnetv2_b0',
            'backbone_pooling':'avg',
            'dropout':          0.1,
            'pretrained':       False,
            'channels':         1,
            'num_labels':       234,
        }
        if config:
            cfg.update(config)
        self.backbone = timm.create_model(
            cfg['backbone'],
            pretrained=cfg['pretrained'],
            num_classes=cfg['num_labels'],
            global_pool=cfg['backbone_pooling'],
            in_chans=cfg['channels'],
            drop_rate=cfg['dropout'],
        )

    def forward(self, x):
        return self.backbone(x)

# ─── Dataset ─────────────────────────────────────────────────────────────────

class BirdDataset(Dataset):
    def __init__(self, paths, spec_transform):
        self.paths = paths
        self.spec  = spec_transform
        self.n_seg = int(60 / DUR)

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        filepath = self.paths[idx]
        try:
            wav, _  = torchaudio.load(filepath)
            wav     = wav.float()[:, :SR * 60]
            wav     = wav.reshape((self.n_seg, SR * DUR))
            mel     = torch.stack([self.spec(wav[i]) for i in range(len(wav))])
            names   = [
                filepath.split('/')[-1].split('.')[0] + '_' + str(i * DUR + DUR)
                for i in range(self.n_seg)
            ]
        except Exception as e:
            print(f"Error loading {filepath}: {e}")
            mel   = torch.zeros((self.n_seg, 1, 256, 256))
            names = [filepath.split('/')[-1].split('.')[0] + '_' + str(i * DUR + DUR)
                     for i in range(self.n_seg)]
        return mel.numpy().astype(np.float32), names   # (n_seg, C, H, W)

# ─── OpenVINO Export & Compile ───────────────────────────────────────────────

def export_to_openvino(ckpt_path, ov_model_path, config=None):
    if os.path.exists(ov_model_path):
        return
    print(f"Exporting {ckpt_path} -> {ov_model_path}")
    state = torch.load(ckpt_path, map_location='cpu', weights_only=True)
    if isinstance(state, dict) and 'model_state_dict' in state:
        state = state['model_state_dict']
    # Infer num_labels directly from checkpoint to avoid size mismatch
    num_labels = state['backbone.classifier.weight'].shape[0]
    cfg = config.copy() if config else {}
    cfg['num_labels'] = num_labels
    model = BirdModel(cfg)
    model.load_state_dict(state)
    model.eval()
    dummy    = torch.zeros(1, 1, 256, 256)
    ov_model = ov.convert_model(model, example_input=dummy)
    ov.save_model(ov_model, ov_model_path)
    print(f"Saved: {ov_model_path}")


def compile_ov_model(ov_model_path):
    core     = ov.Core()
    ov_model = core.read_model(ov_model_path)
    compiled = core.compile_model(ov_model, device_name=ov_device)
    return compiled

# ─── Inference ───────────────────────────────────────────────────────────────

def predict_batch(compiled_models, batch_np):
    """batch_np: (B, C, H, W)"""
    preds = []
    for compiled in compiled_models:
        logits = compiled.infer_new_request({0: batch_np})[compiled.output(0)]
        probs  = 1.0 / (1.0 + np.exp(-logits))
        preds.append(probs)
    return np.mean(preds, axis=0)


def run_inference(compiled_models, paths, spec, num_labels, labels):
    dataset  = BirdDataset(paths, spec)
    all_preds, all_names = [], []

    for mel_segs, names in tqdm(dataset, desc="Infer"):
        # mel_segs: (n_seg, C, H, W)
        preds = predict_batch(compiled_models, mel_segs)   # (n_seg, num_labels)
        all_preds.append(preds)
        all_names.extend(names)

    return np.concatenate(all_preds, axis=0), all_names

# ─── Submission ───────────────────────────────────────────────────────────────

def create_submission(preds, names, all_labels, train_labels):
    n_pred_cols = preds.shape[1]
    # train_labels may be a subset; use only as many as the model predicted
    used_labels = train_labels[:n_pred_cols]
    df = pd.DataFrame(np.zeros((len(preds), len(all_labels))), columns=all_labels)
    df[used_labels] = preds
    df.insert(0, 'row_id', names)
    df.to_csv(SUBMISSION_FILE, index=False)

    print(f"---> Submission shape = {df.set_index("row_id").shape}")
    return df

# ─── Main ─────────────────────────────────────────────────────────────────────

def main():
    taxonomy_df  = pd.read_csv(taxonomy_csv)
    all_labels   = sorted(taxonomy_df['primary_label'].unique().tolist())
    train_labels = sorted(pd.read_csv(PATH + 'train.csv')['primary_label'].unique().tolist())
    num_labels   = len(train_labels)

    paths = [TEST_PATH + x for x in os.listdir(TEST_PATH) if x.endswith('.ogg')]
    if not paths:
        paths = sorted([TRAIN_PATH + x for x in os.listdir(TRAIN_PATH) if x.endswith('.ogg')])[:16]

    print(f"Files: {len(paths)}, Classes: {num_labels}, Device: {ov_device}")

    spec = Spectrogram(sr=SR, n_fft=2048, n_mels=256, hop_length=512,
                       f_min=20, f_max=16000, channels=1,
                       target_size=(256, 256), top_db=80.0)

    ckpt_paths = [os.path.join(model_dir, f)
                  for f in os.listdir(model_dir) if f.endswith('.pth')]

    ov_export_dir = "/kaggle/working/ov_models"
    os.makedirs(ov_export_dir, exist_ok=True)

    compiled_models = []
    for ckpt in ckpt_paths:
        ov_name = os.path.splitext(os.path.basename(ckpt))[0] + '.xml'
        ov_path = os.path.join(ov_export_dir, ov_name)
        export_to_openvino(ckpt, ov_path)
        compiled_models.append(compile_ov_model(ov_path))

    print(f"Loaded {len(compiled_models)} OpenVINO model(s)")

    preds, names = run_inference(compiled_models, paths, spec, num_labels, train_labels)
    create_submission(preds, names, all_labels, train_labels)


if __name__ == "__main__":
    main()

# **SUBMISSION**

In [ ]:
import os, sys, time 
import pandas as pd

!python myinfer.py
print()

print("\n\n\n")
display(
    pd.read_csv("submission.csv", index_col = "row_id").
    head(5)
)